# 课程 01 - AI 代理简介

欢迎来到 **AI 新手代理** 课程的第一课！

**AI 代理** 是一个使用大型语言模型（LLM）作为推理引擎的程序，并且能够在现实世界中采取<em>行动</em> —— 调用 API、查询数据库或运行代码 —— 以代表用户完成目标。

在本笔记本中，您将构建第一个代理：一个推荐度假目的地的 <strong>旅行代理</strong>。在此过程中，您将学习如何：

1. 使用 **Microsoft Agent Framework** 连接到 Microsoft Foundry Agent 服务。
2. 给代理一个 <strong>工具</strong> —— 一个它可以调用的普通 Python 函数。
3. 运行代理并检查其响应。
4. 逐个令牌流式传输代理的响应。


## 设置

在运行此笔记本之前，请确保您已完成以下操作：

1. **拥有一个 Microsoft Foundry 项目** 并已部署聊天模型（例如 `gpt-5-mini`）。
2. **已使用 Azure CLI 登录** — 在终端运行 `az login`。
3. **设置必需的环境变量：**
   - `LLM_BASE_URL` — 您的 Microsoft Foundry 项目端点。
   - `LLM_MODEL` — 您已部署模型的名称。

下面的单元格将安装您需要的 Python 包。


In [15]:
%pip install agent-framework -q

4426.30s - pydevd: Sending message related to process being replaced timed-out after 5 seconds



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.openai import OpenAIChatCompletionClient
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("LLM_BASE_URL")
model = os.getenv("LLM_MODEL")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set LLM_BASE_URL and LLM_MODEL in your .env file."
    )

provider = OpenAIChatCompletionClient(
    model=os.environ["LLM_MODEL"],
    api_key=os.environ["LLM_API_KEY"],
    base_url=os.environ["LLM_BASE_URL"],
)

## 创建你的第一个智能体

一个智能体需要两样东西：

- <strong>指令</strong>，告诉它<em>它是谁</em>以及<em>如何表现</em>（系统提示）。
- <strong>工具</strong> —— 用 `@tool` 装饰的 Python 函数，智能体可以调用它们来获取信息或执行操作。

下面我们定义了一个简单的工具，它返回一个受欢迎的度假目的地列表。当用户询问旅行推荐时，智能体将使用此工具。


In [9]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [12]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "你是一个有用的旅行代理。根据用户的偏好帮他们找到完美的度假目的地。"
        "请使用 get_destinations 工具查看可用目的地列表，并始终用中文回复。"
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "我想去一个温暖的海滩度假地，你推荐哪里？"
)
print(response)

根据可用的度假目的地列表，针对您想要的**温暖海滩度假地**，我最推荐以下几个地方：

## 🏝️ **巴厘岛（Bali）** — 最推荐！
巴厘岛是完美的热带海滩天堂，全年气候温暖，拥有迷人的白沙滩、碧蓝海水和世界级度假村。这里不仅有绝美的海滩，还有独特的文化和放松的氛围，非常适合度假放松。

## 🏖️ **里约热内卢（Rio de Janeiro）**
巴西著名的海滨城市，拥有 iconic 的科帕卡巴纳海滩和伊帕内玛海滩。这里阳光充足、气候温暖，海滩文化浓厚，非常适合喜欢热闹氛围和阳光沙滩的旅客。

## 🌊 **巴塞罗那（Barcelona）**
如果您喜欢地中海风情，巴塞罗那既有美丽的海滩又有丰富的历史文化。这里气候温暖宜人，可以在海滩晒太阳，也可以探索高迪建筑和美食。

---

**个人建议**：如果您纯粹想要温暖的热带海滩体验，**巴厘岛**会是最佳选择；如果您喜欢充满活力的海滩文化和城市风情，**里约热内卢**也非常棒！

您更倾向于哪种风格的海滩度假呢？我可以为您提供更具体的建议。


## 流式响应

为了获得更互动的体验，您可以<strong>流式</strong>获取代理的响应。代理会随着文本生成逐块产出，而不是等待完整回复。这在聊天界面中特别有用，因为您希望实时展示输出内容。


In [13]:
async for chunk in agent.run(
    "请用中文介绍一下东京作为旅行目的地", stream=True
):
    print(chunk, end="", flush=True)

东京是日本的首都，也是全球最繁华、最现代化的都市之一，同时保留了深厚的传统文化底蕴。以下是东京作为旅行目的地的详细介绍：

## 🏙️ 城市概况
东京是世界上最大的都市圈之一，融合了超现代的高科技城市景观与历史悠久的寺庙神社。这里既有霓虹闪烁的摩天大楼，也有宁静雅致的日式庭院，形成了独特的"和洋交融"魅力。

## ⛩️ 必游景点
- **浅草寺与雷门**：东京最古老的寺庙，周围保留着浓厚的江户风情
- **明治神宫**：位于市中心的巨大神社，被大片森林环绕，是感受日本神道文化的好去处
- **涩谷十字路口**：世界上最繁忙的人行横道，象征东京的活力与现代感
- **东京塔 / 东京晴空塔**：俯瞰全城夜景的绝佳地点
- **皇居东御苑**：日本天皇居所的开放花园，四季景色宜人

## 🍣 美食天堂
东京拥有世界上最多的米其林星级餐厅：
- **寿司与刺身**：筑地市场（丰洲市场）的新鲜海味不容错过
- **拉面**：一兰、一风堂等，从浓郁豚骨到清爽酱油风味应有尽有
- **和牛烧肉**：入口即化的顶级牛肉体验
- **居酒屋文化**：体验日本下班后的小酌氛围与串烧、下酒菜

## 🛍️ 购物与潮流
- **银座**：高端奢侈品与百年老店的聚集地
- **涩谷、原宿**：日本年轻人的潮流发源地，街头文化浓厚
- **秋叶原**：动漫、游戏与电子产品的圣地
- **新宿**：24小时不眠的娱乐购物区

## 🌸 最佳旅行季节
- **春季（3-5月）**：樱花季，全城粉色浪漫
- **秋季（9-11月）**：红叶与舒适的气候
- **冬季**：可体验日本新年传统与美食

## 🚇 交通提示
东京的铁路网络极其发达（JR、地铁、私铁交织），建议购买 **Suica / Pasmo 交通卡** 或 **东京地铁通票**。虽然线路复杂，但标识清晰，大部分有中文提示。

## 💡 适合谁去？
东京非常适合**首次赴日游客**、**美食爱好者**、**购物达人**、**动漫文化爱好者**，以及想要体验**现代都市与传统文化并存**的旅行者。

您是否有特定的旅行偏好（比如美食、购物、文化或亲子游）？我可以为您提供更针对性的建议！

## 总结

在本课中，您学到了如何：

- <strong>创建一个提供程序</strong>，通过 `FoundryChatClient` 连接到 Microsoft Foundry Agent Service。
- **使用 `@tool` 装饰器定义工具**，以便代理可以调用您的 Python 函数。
- <strong>运行代理</strong>，发送用户消息并打印其响应。
- <strong>流式传输响应</strong>，实现实时输出。

在下一课中，我们将更深入地探讨代理框架，并学习如何赋予代理更强大的工具和多步骤推理能力。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
